<img src="https://www.th-koeln.de/img/logo.svg" style="float:right;" width="200">

# 14th exercise: <font color="#C70039">State representation in reinforcement learning</font>

* Course: AML  
* Lecturer: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Author of notebook: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Date: 03.09.2026

<img src="./assets/evolution.png" width="400">

---

**GENERAL NOTE 1**: 
Please make sure you are reading the entire notebook, since it contains a lot of information on your tasks (e.g. regarding the set of certain parameters or a specific computational trick), and the written mark downs as well as comments contain a lot of information on how things work together as a whole. 

**GENERAL NOTE 2**: 
* Please, when commenting source code, just use English language only. 
* When describing an observation please use English language, too.
* This applies to all exercises throughout this course.

---------------------------------

### <font color="FFC300">LEARNING OBJECTIVES</font>:

After this exercise, you can explain why a state representation must contain all decision-relevant information, compare policies learned with different state spaces, interpret a Q-table as a policy, and reason about reward design.

## Position in the reinforcement-learning sequence

Exercise 12 introduced tabular Q-learning on a deterministic graph. Exercise 13 transferred the same algorithm to a stochastic Gymnasium environment. This exercise uses a custom Gymnasium environment, but **the Q-learning implementation is provided**. The focus is not another implementation of the training loop - it is the design of the state space.

A state is adequate only if it contains the information needed to choose an optimal action. This requirement is closely related to the Markov property. The future reward distribution should depend on the current state and action, not on hidden history.

## The Evolutionv2 environment

Pikachu starts in the bottom-left corner of a 3×3 grid. The available actions are left, right, up, and down.

- The TM is in the top-left corner. Collecting it yields **+1** and lets Pikachu defeat the enemy.
- The enemy is in the top-right corner. Without the TM it gives **−10**; with the TM it gives **+6**. Both cases end the episode.
- The Thunderstone is in the bottom-right corner. It gives **+5** and ends the episode.
- Walking into an outer wall gives **−1**.

The TM-plus-enemy route has total reward +7 and is therefore better than the Thunderstone route. However, it can only be learned reliably when the agent can distinguish whether it already owns the TM.

## Setup

Install the requirements in this exercise directory once when working locally. The environment implementation and its optional graphics are provided in `Evolution.py` and `assets/`. Rendering is disabled during training because it substantially slows down the learning process.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Support both a repository-root kernel and an exercise-directory kernel.
exercise_directory = Path.cwd() / "Ex14.ReinforcementLearning"
if not (exercise_directory / "Evolution.py").exists():
    exercise_directory = Path.cwd()
sys.path.insert(0, str(exercise_directory))

from Evolution import Evolutionv2

print(f"Using exercise directory: {exercise_directory}")

ModuleNotFoundError: No module named 'gymnasium'

## Two possible state representations

With `all_states=False`, the observation is only Pikachu's grid position. There are 9 states.

With `all_states=True`, the observation is the grid position plus the Boolean TM status. There are 18 states: nine positions without the TM and the same nine positions after collecting it.

The same physical position can lead to different optimal decisions depending on TM status. A position-only observation hides that distinction.

In [ ]:
position_environment = Evolutionv2(render_mode=None, all_states=False)
augmented_environment = Evolutionv2(render_mode=None, all_states=True)

print("Position-only states:", position_environment.observation_space.n)
print("Position-plus-TM states:", augmented_environment.observation_space.n)
print("Actions:", position_environment.action_space.n)

position_environment.close()
augmented_environment.close()

## Provided Q-learning training function

Read this function and connect it to the implementation you created in Exercises 12 and 13. Notice the terminal-state handling and the epsilon-greedy policy. Do not modify the function for the initial comparison; change only the state representation through its `all_states` argument.

In [ ]:
def train_agent(all_states, seed=1, num_episodes=4_000, max_steps=50):
    environment = Evolutionv2(render_mode=None, all_states=all_states)
    environment.action_space.seed(seed)
    random_generator = np.random.default_rng(seed)
    q_table = np.zeros((environment.observation_space.n, environment.action_space.n))

    learning_rate = 0.3
    discount_rate = 0.95
    exploration_rate = 1.0
    min_exploration_rate = 0.01
    exploration_decay_rate = 0.0015
    episode_rewards = []

    for episode in range(num_episodes):
        state, info = environment.reset(seed=seed + episode)
        total_reward = 0.0

        for _ in range(max_steps):
            if random_generator.random() < exploration_rate:
                action = environment.action_space.sample()
            else:
                action = int(np.argmax(q_table[state]))

            new_state, reward, terminated, truncated, info = environment.step(action)
            episode_finished = terminated or truncated
            future_value = 0.0 if episode_finished else np.max(q_table[new_state])
            temporal_difference = reward + discount_rate * future_value - q_table[state, action]
            q_table[state, action] += learning_rate * temporal_difference

            state = new_state
            total_reward += reward
            if episode_finished:
                break

        exploration_rate = min_exploration_rate + (1.0 - min_exploration_rate) * np.exp(
            -exploration_decay_rate * episode
        )
        episode_rewards.append(total_reward)

    environment.close()
    return q_table, np.asarray(episode_rewards)

## <font color="FFC300">Task 1</font> — compare learning outcomes

Train both variants with the same seed. Compare the mean reward in the final 500 episodes. Before running the cell, write a prediction: which representation should learn the higher-reward TM-plus-enemy route, and why?

Do not infer a policy only from the reward average. Use the Q-table visualisation and greedy evaluation below as supporting evidence.

In [ ]:
q_position, rewards_position = train_agent(all_states=False)
q_augmented, rewards_augmented = train_agent(all_states=True)

print(f"Position only — final 500 episodes: {rewards_position[-500:].mean():.3f}")
print(f"Position + TM  — final 500 episodes: {rewards_augmented[-500:].mean():.3f}")

plt.figure(figsize=(10, 4))
plt.plot(rewards_position, alpha=0.35, label="Position only")
plt.plot(rewards_augmented, alpha=0.35, label="Position + TM")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.legend()
plt.show()

## Q-table visualisation

Each number is the estimated return for one action at one grid cell. The first image represents states without the TM. If the augmented representation is used, the second image represents the same positions after the TM was collected.

Inspect the action values near the TM, enemy, and Thunderstone. Identify an action whose value changes after the TM is collected.

In [ ]:
def show_q_tables(q_table, all_states):
    environment = Evolutionv2(render_mode=None, all_states=all_states)
    state_layers = [1, 2] if all_states else [1]

    plt.figure(figsize=(8 * len(state_layers), 6))
    for index, state_layer in enumerate(state_layers, start=1):
        plt.subplot(1, len(state_layers), index)
        plt.imshow(environment._get_q_frame_info(q_table, state_layer))
        plt.axis("off")
        plt.title("Without TM" if state_layer == 1 else "With TM")
    plt.show()
    environment.close()

show_q_tables(q_position, all_states=False)
show_q_tables(q_augmented, all_states=True)

## Greedy-policy evaluation

The next function runs the policy without exploratory actions. It reports the frequency of each terminal outcome. The result is more informative than a single average reward: it shows whether the learned policy aims for the Thunderstone, defeats the enemy, or still loses to the enemy.

Run it for both Q-tables and explain the difference.

In [ ]:
def evaluate_policy(q_table, all_states, episodes=500):
    environment = Evolutionv2(render_mode=None, all_states=all_states)
    outcome_counts = {"enemy defeated": 0, "enemy loss": 0, "thunderstone": 0, "timeout": 0}

    for episode in range(episodes):
        state, info = environment.reset(seed=10_000 + episode)
        for _ in range(50):
            action = int(np.argmax(q_table[state]))
            state, reward, terminated, truncated, info = environment.step(action)
            if terminated or truncated:
                if reward == 6:
                    outcome_counts["enemy defeated"] += 1
                elif reward == -10:
                    outcome_counts["enemy loss"] += 1
                elif reward == 5:
                    outcome_counts["thunderstone"] += 1
                else:
                    outcome_counts["timeout"] += 1
                break
        else:
            outcome_counts["timeout"] += 1

    environment.close()
    return outcome_counts

print("Position only:", evaluate_policy(q_position, all_states=False))
print("Position + TM:", evaluate_policy(q_augmented, all_states=True))

## <font color="FFC300">Task 2</font> — explain the state-design result

Write a short Markdown answer addressing all points:

1. Which two histories can lead to the same position but require different decisions?
2. Why does this make a position-only state representation insufficient?
3. Why are 18 states sufficient in this environment?
4. How do the Q-table images support your explanation?

Use the terms **state representation**, **hidden history**, **policy**, and **Markov property** correctly.

## <font color="FFC300">Task 3</font> — reward-design experiment

Change exactly one reward in `Evolution.py` for a controlled experiment. For example, change the reward for defeating the enemy from +6 to +4 or +9. Re-run both representations, report the outcome frequencies, and explain how the reward change alters the preferred policy.

Restore the original value after documenting your result. Do not change multiple rewards at the same time.

## Optional visual run

`qexample.py` and `qexamplev2.py` are optional instructor reference runners. They open the graphical environment and are not required for this exercise. Use them only after the notebook analysis is complete; the visual rendering slows down the training substantially.

## <font color="FFC300">Task 4</font> — Reflection

1. Why is a visually rich environment not automatically a richer observation for the agent?
2. What extra variable would be needed if the TM could be lost again?
3. Which learning problem in this notebook would a neural-network function approximator address, and which one would it not solve by itself?